# OpenPlaque — Secondary Branch Persistence

Freeze the validated LAD backbone, the strong takeoff candidate, and the previously passed 8.8 mm proximal trunk. Search only for one independent coronary-like secondary branch that persists 6–12 mm while separating from the established LAD/trunk reference. No global LAD search, no trunk rediscovery, and no automatic LCX label. Research use only.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Reuse controls — True + valid cache reuses; False forces recomputation/overwrite.
REUSE_SOURCE_CT = True
REUSE_FROZEN_GEOMETRY = True
REUSE_BRANCH_SEARCH = True
REUSE_FIGURES = True
REUSE_REPORT = True


## Step 3 — Install dependencies


In [ ]:
%pip -q install scipy matplotlib pandas
print('Dependencies ready.')


## Step 4 — Load this fresh branch


In [ ]:
import os, sys, subprocess, shutil
REPO='/content/OpenPlaque'
BRANCH='secondary-branch-persistence-from-main'
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(['git','clone','-q','--depth','1','--branch',BRANCH,'https://github.com/pazzani/OpenPlaque.git',REPO],check=True)
sys.path.insert(0,os.path.join(REPO,'src'))
print('Loaded',BRANCH)


## Step 5 — Initialize and inspect caches


In [ ]:
from openplaque.secondary_branch_persistence import SecondaryBranchPersistenceWorkflow
reuse={
 'source_ct':REUSE_SOURCE_CT,
 'frozen_geometry':REUSE_FROZEN_GEOMETRY,
 'branch_search':REUSE_BRANCH_SEARCH,
 'figures':REUSE_FIGURES,
 'report':REUSE_REPORT,
}
wf=SecondaryBranchPersistenceWorkflow(reuse=reuse)
display(wf.cache_status())


## Step 6 — Load source CCTA and freeze prior geometry

This imports the existing disk-backed source CCTA, alternative 1, the takeoff candidate, and the previously passed proximal trunk. The workflow refuses to continue if that trunk was not accepted.


In [ ]:
wf.load_source_ct()
frozen=wf.load_frozen_geometry()
print('Frozen candidate:', frozen['candidate'])
print('Prior trunk:', frozen['trunk_summary'])


## Step 7 — Trace secondary branch only

The beam search explicitly rewards separation from the established LAD/trunk reference while preserving source-resolution coronary-lumen QC. It searches up to 12 mm and keeps multiple alternatives before selecting the best persistent branch.


In [ ]:
summary=wf.search_branch(max_length_mm=12.0,beam_width=24)
print(summary)
if wf.candidates is not None and len(wf.candidates): display(wf.candidates.head(20))


## Step 8 — Generate QC figures


In [ ]:
figs=wf.plot_qc()
from IPython.display import display, Image
for f in figs:
    print(f)
    display(Image(filename=str(f)))


## Step 9 — Package report

A PASS means the candidate supports a persistent independent coronary-like secondary branch. It still does not automatically label the branch LCX.


In [ ]:
z=wf.package()
print('REPORT_BACK:',z)
print('Final status:',wf.summary.get('status'))
